In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
print(spark.version)


3.5.5


In [3]:
spark.sql("SELECT * FROM demo.smoke.t ORDER BY id").show()


+---+----------------+
| id|             msg|
+---+----------------+
|  1| hello lakehouse|
|  2|iceberg on minio|
+---+----------------+



In [4]:
spark.sql("SELECT snapshot_id, operation FROM demo.smoke.t.snapshots").show(truncate=False)


+-------------------+---------+
|snapshot_id        |operation|
+-------------------+---------+
|2278561666384766099|append   |
+-------------------+---------+



In [5]:
spark.sql("SELECT file_path, record_count FROM demo.smoke.t.files").show(truncate=False)


+----------------------------------------------------------------------------------------+------------+
|file_path                                                                               |record_count|
+----------------------------------------------------------------------------------------+------------+
|s3://warehouse/smoke/t/data/00000-0-30018625-f701-4da0-a1aa-490292147e3d-0-00001.parquet|1           |
|s3://warehouse/smoke/t/data/00001-1-30018625-f701-4da0-a1aa-490292147e3d-0-00001.parquet|1           |
+----------------------------------------------------------------------------------------+------------+



In [7]:
spark.sql("INSERT INTO demo.smoke.t VALUES (3,'third')")
print("snapshots:")
spark.sql("SELECT snapshot_id, operation FROM demo.smoke.t.snapshots").show(truncate=False)
print("files:")
spark.sql("SELECT file_path, record_count FROM demo.smoke.t.files").show(truncate=False)


snapshots:
+-------------------+---------+
|snapshot_id        |operation|
+-------------------+---------+
|2278561666384766099|append   |
|8627637303347187883|append   |
|1595117439452203802|append   |
+-------------------+---------+

files:
+----------------------------------------------------------------------------------------+------------+
|file_path                                                                               |record_count|
+----------------------------------------------------------------------------------------+------------+
|s3://warehouse/smoke/t/data/00000-6-36e07b9b-5e42-4b95-81bd-e6d48908e2ee-0-00001.parquet|1           |
|s3://warehouse/smoke/t/data/00000-3-406ef758-7b51-4cd3-bc3c-41d37c73afc6-0-00001.parquet|1           |
|s3://warehouse/smoke/t/data/00000-0-30018625-f701-4da0-a1aa-490292147e3d-0-00001.parquet|1           |
|s3://warehouse/smoke/t/data/00001-1-30018625-f701-4da0-a1aa-490292147e3d-0-00001.parquet|1           |
+----------------------------

In [8]:
spark.sql("SELECT * FROM demo.smoke.t VERSION AS OF 2278561666384766099").show()  # 2 rows

+---+----------------+
| id|             msg|
+---+----------------+
|  1| hello lakehouse|
|  2|iceberg on minio|
+---+----------------+



In [10]:
spark.sql("SELECT * FROM demo.smoke.t VERSION AS OF 8627637303347187883").show()  # 3 rows

+---+----------------+
| id|             msg|
+---+----------------+
|  1| hello lakehouse|
|  2|iceberg on minio|
|  3|           third|
+---+----------------+



In [17]:
spark.sql("SELECT committed_at, snapshot_id, operation FROM demo.smoke.t.snapshots").show(truncate=False)


+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2026-06-14 11:41:56.681|2278561666384766099|append   |
|2026-06-14 13:12:06.106|8627637303347187883|append   |
|2026-06-14 13:21:39.846|1595117439452203802|append   |
+-----------------------+-------------------+---------+



In [20]:
spark.sql("SELECT * FROM demo.smoke.t.snapshots ").show()


+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2026-06-14 11:41:...|2278561666384766099|               NULL|   append|s3://warehouse/sm...|{spark.app.id -> ...|
|2026-06-14 13:12:...|8627637303347187883|2278561666384766099|   append|s3://warehouse/sm...|{spark.app.id -> ...|
|2026-06-14 13:21:...|1595117439452203802|8627637303347187883|   append|s3://warehouse/sm...|{spark.app.id -> ...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+

